# knob-senses — servidor de teste (CDC serial)

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, Path.cwd().parent / 'models' / 'src')
from common import contract, data, paths, gaps, server

AXES = server.AXES

HZ = 1.0

# Um stdout REDIRECIONADO (pipe, arquivo, `jupyter nbconvert --execute`) no
# Windows cai no codepage da locale (cp1252), e ai o travessao do log vira o
# byte 0x97 — que nao e UTF-8 valido. Quem le o pipe com utf-8 quebra no meio.
# Num console interativo ou no Jupyter o Python ja acerta; isto e para o resto.
# O OutStream do Jupyter nao tem reconfigure(), dai o AttributeError no guard.
for _s in (sys.stdout, sys.stderr):
    try:
        _s.reconfigure(encoding="utf-8", errors="replace")
    except (AttributeError, ValueError):
        pass


# Granularidade do ensaio: o split de teste mistura 5min/10min/1h/1D (coluna
# `freq`), e o firmware do modelo treinado so' faz sentido na granularidade
# dele. O `seq` do protocolo conta as linhas DESTE filtro.
FREQ_ENSAIO = "1h"   # "5min" | "10min" | "1h" | "1D"
assert FREQ_ENSAIO in contract.GRANULARIDADES, \
    f"FREQ_ENSAIO={FREQ_ENSAIO} fora de {contract.GRANULARIDADES}"

DATA_SOURCE = paths.SPLITS / "test"
print(f"dados em {DATA_SOURCE} | granularidade {FREQ_ENSAIO}", flush=True)
print(f"resultados em {paths.RESULTS}", flush=True)

dados em C:\Users\gusta\Documents\PlatformIO\Projects\knob-senses\data\splits\test | granularidade 1h
resultados em C:\Users\gusta\Documents\PlatformIO\Projects\knob-senses\results


In [2]:
# As copias por inversor (2025 de cada usina), filtradas pela granularidade do
# ensaio. Um serve() roda por inversor e grava results/ensaio/{inversor}_val.csv.
import pandas as pd

df = pd.concat(
   (data.read_csv(f) for f in (paths.SPLITS / "test").rglob('*')), 
   ignore_index=True
)


df = df[df["freq"] == FREQ_ENSAIO].reset_index(drop=True)

df[['inversor', *contract.RF_FEATURES]].groupby('inversor').describe(percentiles=[])


temp_amb                                               ghit  \
            count       mean       std        min        max   count   
inversor                                                               
ifpee1     4042.0  27.355046  4.282753  15.160667  37.317667  4042.0   
ifpee2     4042.0  27.355370  4.282948  15.160667  37.317667  4042.0   
ifpee3     4042.0  27.355154  4.282877  15.160667  37.317667  4042.0   

                                                        jacques             \
                mean         std       min          max   count       mean   
inversor                                                                     
ifpee1    521.760069  319.771220  6.550885  1250.213240  4042.0  41.322786   
ifpee2    521.880922  319.947728  6.550885  1247.626867  4042.0  41.326346   
ifpee3    521.709184  319.747020  6.550885  1247.626867  4042.0  41.321532   

                                           
                std        min        max  
inversor                                   
ifpee1    11.942459  17.194545  68.980127  
ifpee2    11.947411  17.194545  68.980127  
ifpee3    11.942139  17.194545  68.980127

In [3]:
for sample in df[df.index < 5].iloc:
    print(server.sample_bytes(sample))

b'\xb4\x8f\xbfAyC\xe7BPS\xd8A'
b',\xf9\xcdA\x83\x1e\xa8C\xdd\xfd\nB'
b'V\x8e\xd9A\xdd\xf7\xdeC\x96\x87\x1cB'
b'y\xe9\xe7A\x86=$D\xf6M:B'
b'\xda\xc0\xf9A\xa6\xb5\x7fDigjB'


In [4]:
import numpy as np
# Resumivel: cada serve_serial() bloqueia ate' o firmware terminar o dataset
# daquela usina. Re-rodar a celula NAO zera o que ja' foi ensaiado: as usinas
# que ja' estao no reply sao puladas, e o ensaio continua de onde parou. Para
# recomecar tudo do zero, execute `reply = dict()` antes.
#
# Os dados passam pela CDC do USB-Serial/JTAG (PORTA_SERIAL); o log do firmware
# fica na UART0/COM28. Feche o OpenOCD antes (o COM29 e' compartilhado).
PORTA_SERIAL = "COM29"  # a CDC da porta "USB" nativa
GAPS = ("24h", "24h", "3d", "10d", "6h", "6h")

SEEDS = (
    np.random
    .default_rng(seed=contract.SEED)
    .integers(low=0, high=128, size=len(df.inversor.unique()))
)

print(f"Random SEEDS={SEEDS}")

def ensaio(args, seed):
    inversor, df = args
    result = server.serve_serial(df, PORTA_SERIAL)
    result["inversor"] = inversor
    result["gap"], result["gap_size"] = gaps.gerar_gaps(df, freq=FREQ_ENSAIO, gaps=GAPS, seed=seed)
    return result

results = pd.concat([ensaio(args, seed) for args, seed in zip(df.groupby('inversor'), SEEDS)])
results.to_csv(paths.ENSAIO / "board_test.csv", index=False)

Random SEEDS=[11 99 83]
Serial aberta em COM29 (baud virtual em USB): sessao iniciada
sample 2025-01-01 06:00:00: real=0.2958, pred=0.1350, err=0.1608, inferencia=4019 us
lote base=0..63 ok (64/4042 linhas)
sample 2025-01-09 10:00:00: real=1.4467, pred=1.5047, err=0.0581, inferencia=978 us
lote base=64..127 ok (128/4042 linhas)
lote base=128..191 ok (192/4042 linhas)
sample 2025-01-17 14:00:00: real=0.7825, pred=0.7360, err=0.0465, inferencia=978 us
lote base=192..255 ok (256/4042 linhas)
sample 2025-01-26 06:00:00: real=0.1200, pred=0.0920, err=0.0280, inferencia=978 us
lote base=256..319 ok (320/4042 linhas)
lote base=320..383 ok (384/4042 linhas)
sample 2025-02-03 10:00:00: real=1.5583, pred=1.5879, err=0.0296, inferencia=978 us
lote base=384..447 ok (448/4042 linhas)
sample 2025-02-11 14:00:00: real=1.1150, pred=1.1778, err=0.0628, inferencia=978 us
lote base=448..511 ok (512/4042 linhas)
lote base=512..575 ok (576/4042 linhas)
sample 2025-02-20 06:00:00: real=0.1187, pred=0.1012, 